# Machine Learning CIE 2 – Activity 1
## Dataset Preparation and Exploratory Data Analysis on Titanic Dataset

**Unit:** Unit I – Foundations & Data Preparation  
**Objective:** Inspect a tabular dataset, identify data-quality issues, perform basic EDA, clean the data, encode categorical features, scale suitable numerical features, and save a prepared CSV file.

**Dataset:** Titanic Dataset  
**Source:** [Seaborn Titanic CSV on GitHub](https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv)  
**File type:** CSV

> Run the notebook from top to bottom. The dataset is downloaded from the source URL when the notebook runs. Outputs such as row counts and missing-value percentages are calculated from the actual downloaded file.

## 1. Import required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2. Select and load the dataset
The Titanic dataset contains passenger information such as passenger class, sex, age, ticket fare, port of embarkation and survival status.

In [ ]:
dataset_url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

df = pd.read_csv(dataset_url)

print("Dataset loaded successfully!")
print("Dataset name: Titanic Dataset")
print("Source:", dataset_url)
print("File type: CSV")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

## 3. Inspect the dataset

In [ ]:
print("First five rows:")
display(df.head())

print("\nLast five rows:")
display(df.tail())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

### Meaning of important columns
- `survived`: Survival status (0 = No, 1 = Yes); possible target for a future prediction task.
- `pclass`: Passenger class (1, 2 or 3).
- `sex`: Passenger sex.
- `age`: Passenger age in years.
- `sibsp`: Number of siblings/spouses aboard.
- `parch`: Number of parents/children aboard.
- `fare`: Passenger fare.
- `embarked`: Port of embarkation.
- `who`: Passenger group (man, woman or child).
- `alone`: Whether the passenger travelled alone.

The dataset also contains some columns that are duplicates, identifiers, derived fields or have substantial missing data. These will be reviewed before preparation.

## 4. Identify numerical and categorical columns

In [ ]:
numerical_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical columns:", numerical_columns)
print("\nCategorical columns:", categorical_columns)

print("\nNumerical summary statistics:")
display(df[numerical_columns].describe())

print("\nCategorical summary:")
display(df[categorical_columns].describe())

## 5. Identify missing values

In [ ]:
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100

missing_report = pd.DataFrame({
    "Missing Values": missing_count,
    "Missing Percentage": missing_percent.round(2)
})

print("Missing-value report:")
display(missing_report.sort_values("Missing Values", ascending=False))

print("\nColumns with no missing values:")
print(missing_report[missing_report["Missing Values"] == 0].index.tolist())

print("\nColumns that need attention:")
print(missing_report[missing_report["Missing Values"] > 0].index.tolist())

### Missing-value observations
The report above shows the exact missing-value count and percentage for every column. In this dataset, `age`, `embarked` and `deck` are among the columns that require attention. The actual values are displayed by the notebook, rather than assumed in advance.

## 6. Make a copy and handle missing values

In [ ]:
# Keep the original data unchanged
original_df = df.copy()
clean_df = df.copy()

# Fill numerical age values with the median.
# Median is less affected by unusually high or low values than the mean.
if "age" in clean_df.columns:
    clean_df["age"] = clean_df["age"].fillna(clean_df["age"].median())

# Fill missing embarked values with the most frequent category (mode).
if "embarked" in clean_df.columns:
    clean_df["embarked"] = clean_df["embarked"].fillna(clean_df["embarked"].mode()[0])

# Deck has a large proportion of missing values and is not required
# for this activity's selected feature set, so it will be removed later.

print("Missing values after filling age and embarked:")
display(clean_df.isnull().sum().to_frame("Remaining Missing Values"))

## 7. Check duplicate records and data consistency

In [ ]:
duplicate_count = clean_df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

# Remove exact duplicate rows, if any.
clean_df = clean_df.drop_duplicates().copy()

# Remove leading/trailing spaces from text columns.
for col in clean_df.select_dtypes(include=["object"]).columns:
    clean_df[col] = clean_df[col].str.strip()

# Standardize the text category values to lowercase for consistency.
for col in clean_df.select_dtypes(include=["object"]).columns:
    clean_df[col] = clean_df[col].str.lower()

print("Duplicate rows after removal:", clean_df.duplicated().sum())
print("\nUnique category values:")
for col in clean_df.select_dtypes(include=["object"]).columns:
    print(col, ":", clean_df[col].dropna().unique()[:20])

print("\nCheck suspicious negative age/fare values:")
if "age" in clean_df.columns:
    print("Negative ages:", (clean_df["age"] < 0).sum())
if "fare" in clean_df.columns:
    print("Negative fares:", (clean_df["fare"] < 0).sum())

### Data-quality decisions
- Exact duplicate rows are removed because they may represent repeated copies of the same observation.
- Extra spaces and inconsistent text case are standardized.
- Unusual values are checked rather than automatically deleted.
- The original dataset remains unchanged in `original_df`.

## 8. Basic exploratory data analysis (EDA)

In [ ]:
print("Summary statistics of numerical columns:")
display(clean_df.describe())

print("\nFrequency counts for passenger class:")
display(clean_df["pclass"].value_counts().sort_index())

print("\nFrequency counts for sex:")
display(clean_df["sex"].value_counts())

### Visualization 1: Age distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=clean_df, x="age", bins=25, kde=True)
plt.title("Distribution of Passenger Ages")
plt.xlabel("Age (years)")
plt.ylabel("Number of Passengers")
plt.tight_layout()
plt.show()

### Visualization 2: Passenger count by class

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=clean_df, x="pclass")
plt.title("Number of Passengers in Each Class")
plt.xlabel("Passenger Class")
plt.ylabel("Number of Passengers")
plt.tight_layout()
plt.show()

### Visualization 3: Fare distribution and outliers

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=clean_df, x="fare")
plt.title("Box Plot of Passenger Fares")
plt.xlabel("Fare")
plt.tight_layout()
plt.show()

### EDA observations
After running the graphs, write observations based on the displayed outputs. Typical points to check:
1. The age histogram shows how passenger ages are distributed and whether most passengers fall within a particular age range.
2. The class count plot compares the number of passengers in first, second and third class.
3. The fare box plot helps identify spread and possible unusually high fare values. Such values should be investigated, not removed automatically.

## 9. Select useful columns and prepare features

In [ ]:
# Keep relevant columns for a future passenger-survival analysis.
# Remove columns that are redundant, identifiers, have substantial missingness,
# or directly reveal the target outcome.
columns_to_drop = [
    "deck",          # substantial missingness
    "class",         # duplicates pclass information
    "embark_town",   # duplicates embarked information
    "alive",         # directly duplicates the target survived
    "adult_male"     # derived/redundant with other passenger information
]

prepared = clean_df.drop(columns=columns_to_drop, errors="ignore").copy()

# Remove text/identifier fields not selected as model features, if present.
other_unselected = ["who", "alone", "sex", "embarked", "pclass",
                    "age", "sibsp", "parch", "fare", "survived"]
extra_drop = [c for c in prepared.columns if c not in other_unselected]
prepared = prepared.drop(columns=extra_drop, errors="ignore")

print("Columns retained before encoding:")
print(prepared.columns.tolist())
display(prepared.head())

## 10. Encode categorical features

In [ ]:
# One-hot encoding is suitable for nominal categories because they have no natural order.
categorical_features = ["sex", "embarked", "who", "alone", "pclass"]
categorical_features = [c for c in categorical_features if c in prepared.columns]

prepared_encoded = pd.get_dummies(
    prepared,
    columns=categorical_features,
    drop_first=False,
    dtype=int
)

print("Categorical columns encoded:", categorical_features)
print("\nDataset after one-hot encoding:")
display(prepared_encoded.head())

print("\nShape after encoding:", prepared_encoded.shape)

### Encoding explanation
Before encoding, columns such as `sex`, `embarked` and `who` contain category labels. One-hot encoding creates separate 0/1 columns for the categories. For example, a passenger belongs to one encoded sex category, represented by 1, while the other category column is 0. Passenger class is also encoded as categories rather than treated as a continuous measurement.

## 11. Scale suitable numerical features

In [ ]:
# Standardization transforms selected numerical input features to approximately
# mean 0 and standard deviation 1. Do not scale the target or identifiers.
numeric_features = ["age", "sibsp", "parch", "fare"]
numeric_features = [c for c in numeric_features if c in prepared_encoded.columns]

scaler = StandardScaler()
prepared_encoded[numeric_features] = scaler.fit_transform(prepared_encoded[numeric_features])

print("Scaling method: StandardScaler (standardization)")
print("Scaled numerical features:", numeric_features)
display(prepared_encoded[numeric_features].describe().round(3))

### Scaling explanation
Age, family counts and fare use different measurement scales. Standardization puts the selected numerical features on a comparable scale, which can help algorithms sensitive to feature magnitude, such as K-nearest neighbours, support vector machines and some optimization-based models.

The `survived` target is not scaled. This notebook prepares the complete dataset for an educational activity and does not train a model. In a real predictive workflow, split the data first and fit the scaler only on the training features; then use that fitted scaler to transform validation/test data.

## 12. Verify and save the prepared dataset

In [ ]:
print("Final dataset shape:", prepared_encoded.shape)
print("\nRemaining missing values:", int(prepared_encoded.isnull().sum().sum()))
print("Remaining duplicate rows:", int(prepared_encoded.duplicated().sum()))

print("\nFinal prepared data sample:")
display(prepared_encoded.head(10))

output_file = "titanic_prepared.csv"
prepared_encoded.to_csv(output_file, index=False)

print("\nPrepared dataset saved successfully as:", output_file)

## 13. Conclusion

The Titanic dataset was inspected to understand its rows, columns and data types. Missing values were identified and suitable methods were used: the median for missing age values and the mode for missing embarkation values. Exact duplicate rows and text consistency were checked. Basic EDA was performed using summary statistics, frequency counts, a histogram, a count plot and a box plot. Redundant or unsuitable columns were removed with reasons, categorical features were one-hot encoded, and selected numerical input features were standardized. The target column `survived` was retained without scaling. Finally, the prepared dataset was saved separately as `titanic_prepared.csv`, leaving the original dataset unchanged.

**No prediction model was trained**, because this activity focuses on dataset inspection and preparation.

## 14. Submission checklist
- [ ] Run all notebook cells and ensure outputs/graphs are visible.
- [ ] Download `titanic_prepared.csv` from the Colab/Jupyter working directory.
- [ ] Submit this notebook and the prepared CSV.
- [ ] Publish the notebook, README and required files in your own public GitHub repository.
- [ ] Add your roll number to the repository name.
- [ ] Check that the public repository URL opens without sign-in or access requests.